# Repetirem tot l'anàlisi d'escala però ara per la T

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from matplotlib import gridspec

# ============================================================
# CONFIG
# ============================================================

CSV_PATH = r"../../data/all_surveys(votes).csv"

#  environmental variable

X_COL = "<T-T_fixed+<T>>"
# X_COL = "<HDX-HDX_fixed+<HDX>>"

COMFORT_COL = "thermal_comfort"

ANALYSIS_MODE = "3_groups_temperature"
# ANALYSIS_MODE = "3_groups_HDX"

# If X_COL = "T_K" and values are Kelvin, convert automatically to Celsius
AUTO_CONVERT_KELVIN = True

# Scale analysis parameters
MIN_N = 30
MIN_CLASS_N = 3
N_OFFSETS_PER_DELTA = 25

# For temperature, these values are usually reasonable
DELTA_MIN = 0.5
DELTA_STEP = 0.25
DELTA_MAX_CAP = 12

# For HDX, you may prefer:
# DELTA_MIN = 1.0
# DELTA_STEP = 0.5
# DELTA_MAX_CAP = 20

# Comfort grouping
comfort_to_group = {
    "Very comfortable": "1. Acceptable / neutral",
    "Comfortable": "1. Acceptable / neutral",
    "Slightly comfortable": "1. Acceptable / neutral",
    "Neutral": "1. Acceptable / neutral",

    "Slightly uncomfortable": "2. Discomfort",
    "Uncomfortable": "2. Discomfort",

    "Very uncomfortable": "3. Extreme discomfort",
}

group_order = [
    "1. Acceptable / neutral",
    "2. Discomfort",
    "3. Extreme discomfort",
]

group_score = {
    "1. Acceptable / neutral": 1,
    "2. Discomfort": 2,
    "3. Extreme discomfort": 3,
}

# preparem les dades per a fer l'analisi

In [ ]:
# ============================================================
# LOAD DATA
# ============================================================

df_votes = pd.read_csv(CSV_PATH)

df_scale = df_votes.copy()

df_scale[X_COL] = pd.to_numeric(df_scale[X_COL], errors="coerce")

x_col_used = X_COL

# Convert Kelvin to Celsius if needed
if AUTO_CONVERT_KELVIN and X_COL == "T_K":
    if df_scale[X_COL].median() > 100:
        df_scale["T_C"] = df_scale[X_COL] - 273.15
        x_col_used = "T_C"
    else:
        x_col_used = X_COL

# 3-group comfort
df_scale["comfort_group"] = df_scale[COMFORT_COL].map(comfort_to_group)
df_scale["comfort_score"] = df_scale["comfort_group"].map(group_score)

df_scale = df_scale.dropna(
    subset=[x_col_used, "comfort_group", "comfort_score"]
).copy()

df_scale = df_scale[df_scale["comfort_group"].isin(group_order)].copy()

print("Environmental variable used:", x_col_used)
print("Analysis mode:", ANALYSIS_MODE)
print("N:", len(df_scale))

print("\nComfort group counts:")
print(df_scale["comfort_group"].value_counts().reindex(group_order))

print("\nEnvironmental variable summary:")
print(df_scale[x_col_used].describe().round(3))

In [ ]:
def analyze_one_tiling_scale(
    df,
    x_col,
    label_col,
    score_col,
    class_order,
    delta_x,
    offset=0.0,
    min_n=30,
    min_class_n=3,
):
    """
    Creates non-overlapping windows of width delta_x.
    The offset shifts the bin origin, allowing many tilings.
    """

    x_min = df[x_col].min()
    x_max = df[x_col].max()

    start = x_min - delta_x + offset
    edges = np.arange(start, x_max + 2 * delta_x, delta_x)

    d = df.copy()

    d["x_window"] = pd.cut(
        d[x_col],
        bins=edges,
        include_lowest=True,
        right=False,
    )

    rows = []

    for window, sub in d.groupby("x_window", observed=False):
        if sub.empty:
            continue

        low = window.left
        high = window.right
        center = 0.5 * (low + high)
        n = len(sub)

        counts = (
            sub[label_col]
            .value_counts()
            .reindex(class_order)
            .fillna(0)
        )

        present_classes = counts[counts > 0].index.tolist()
        usable_classes = counts[counts >= min_class_n].index.tolist()

        p_counts = counts[counts > 0] / counts.sum()
        entropy = -(p_counts * np.log(p_counts)).sum()

        valid_test = (n >= min_n) and (len(usable_classes) >= 2)

        H = np.nan
        p_kw = np.nan
        epsilon2 = np.nan
        rho = np.nan
        p_spear = np.nan
        mean_spread = np.nan
        spread_over_iqr = np.nan
        spread_over_sd = np.nan

        if valid_test:
            groups = [
                sub.loc[sub[label_col] == c, x_col].dropna().values
                for c in usable_classes
            ]

            H, p_kw = stats.kruskal(*groups)

            N = sum(len(g) for g in groups)
            k = len(groups)

            epsilon2 = (H - k + 1) / (N - k) if N > k else np.nan

            rho, p_spear = stats.spearmanr(
                sub[x_col],
                sub[score_col],
                nan_policy="omit",
            )

            means = (
                sub.groupby(label_col, observed=False)[x_col]
                .mean()
                .reindex(class_order)
            )

            means = means.loc[
                [c for c in means.index if counts.get(c, 0) >= min_class_n]
            ].dropna()

            if len(means) >= 2:
                mean_spread = means.max() - means.min()

                x_iqr = sub[x_col].quantile(0.75) - sub[x_col].quantile(0.25)
                x_sd = sub[x_col].std()

                spread_over_iqr = mean_spread / x_iqr if x_iqr > 0 else np.nan
                spread_over_sd = mean_spread / x_sd if x_sd > 0 else np.nan

        rows.append({
            "delta_x": delta_x,
            "offset": offset,
            "window": str(window),
            "low": low,
            "high": high,
            "center": center,
            "n": n,
            "n_present_classes": len(present_classes),
            "n_usable_classes": len(usable_classes),
            "entropy": entropy,
            "valid_test": valid_test,
            "kw_H": H,
            "kw_p": p_kw,
            "epsilon2": epsilon2,
            "spearman_rho": rho,
            "spearman_p": p_spear,
            "mean_spread": mean_spread,
            "spread_over_iqr": spread_over_iqr,
            "spread_over_sd": spread_over_sd,
        })

    return pd.DataFrame(rows)

In [ ]:
# ============================================================
# RUN SCALE ANALYSIS
# ============================================================

x_range = df_scale[x_col_used].max() - df_scale[x_col_used].min()

delta_values = np.arange(
    DELTA_MIN,
    min(0.95 * x_range, DELTA_MAX_CAP) + 0.001,
    DELTA_STEP,
)

all_results = []

for delta in delta_values:
    offsets = np.linspace(0, delta, N_OFFSETS_PER_DELTA, endpoint=False)

    for offset in offsets:
        out = analyze_one_tiling_scale(
            df=df_scale,
            x_col=x_col_used,
            label_col="comfort_group",
            score_col="comfort_score",
            class_order=group_order,
            delta_x=delta,
            offset=offset,
            min_n=MIN_N,
            min_class_n=MIN_CLASS_N,
        )

        all_results.append(out)

scale_offsets = pd.concat(all_results, ignore_index=True)

print("scale_offsets shape:", scale_offsets.shape)
scale_offsets.head(10)
scale_offsets_T = scale_offsets.copy()

In [ ]:
# ============================================================
# SUMMARY BY SCALE
# ============================================================

valid = scale_offsets[scale_offsets["valid_test"]].copy()

valid["minus_log10_p"] = -np.log10(
    valid["kw_p"].clip(lower=1e-12)
)

scale_summary = (
    valid
    .groupby("delta_x")
    .agg(
        n_windows=("kw_p", "count"),

        frac_kw_significant=("kw_p", lambda x: np.mean(x < 0.05)),

        mean_kw_p=("kw_p", "mean"),
        median_kw_p=("kw_p", "median"),
        q25_kw_p=("kw_p", lambda x: np.nanquantile(x, 0.25)),
        q75_kw_p=("kw_p", lambda x: np.nanquantile(x, 0.75)),

        mean_minus_log10_p=("minus_log10_p", "mean"),
        median_minus_log10_p=("minus_log10_p", "median"),
        q25_minus_log10_p=("minus_log10_p", lambda x: np.nanquantile(x, 0.25)),
        q75_minus_log10_p=("minus_log10_p", lambda x: np.nanquantile(x, 0.75)),

        median_kw_H=("kw_H", "median"),
        mean_kw_H=("kw_H", "mean"),

        median_epsilon2=("epsilon2", "median"),
        mean_epsilon2=("epsilon2", "mean"),

        median_spearman=("spearman_rho", "median"),
        mean_spearman=("spearman_rho", "mean"),
        q25_spearman=("spearman_rho", lambda x: np.nanquantile(x, 0.25)),
        q75_spearman=("spearman_rho", lambda x: np.nanquantile(x, 0.75)),

        median_abs_spearman=("spearman_rho", lambda x: np.nanmedian(np.abs(x))),
        frac_positive_spearman=("spearman_rho", lambda x: np.mean(x > 0)),
    )
    .reset_index()
)

scale_summary.round(4)
scale_summary.to_csv("csvs/scale_summary.csv", index=False)
scale_summary_T = scale_summary.copy()

# 2x2 plot rellevant

In [ ]:
fig, axes = plt.subplots(
    2, 2,
    figsize=(13, 9),
    constrained_layout=True,
)

# -----------------------------
# A. Fraction significant
# -----------------------------
ax = axes[0, 0]
ax.plot(
    scale_summary["delta_x"],
    scale_summary["frac_kw_significant"],
    marker="o",
)
ax.axhline(0.5, linestyle="--", alpha=0.7, label="50%")
ax.set_xlabel(f"{x_col_used} window width, Δx")
ax.set_ylabel("Fraction Kruskal p < 0.05")
ax.set_title("A. Statistical detectability")
ax.grid(True, alpha=0.3)
ax.legend()

# -----------------------------
# B. Median -log10(p)
# -----------------------------
ax = axes[0, 1]
x = scale_summary["delta_x"]
y = scale_summary["median_minus_log10_p"]
lo = scale_summary["q25_minus_log10_p"]
hi = scale_summary["q75_minus_log10_p"]

ax.plot(x, y, marker="o", label="median")
ax.fill_between(x, lo, hi, alpha=0.2, label="IQR")
ax.axhline(-np.log10(0.05), linestyle="--", alpha=0.7, label="p = 0.05")
ax.set_xlabel(f"{x_col_used} window width, Δx")
ax.set_ylabel("Median -log10(Kruskal p)")
ax.set_title("B. Strength of Kruskal evidence")
ax.grid(True, alpha=0.3)
ax.legend()

# -----------------------------
# C. Epsilon squared
# -----------------------------
ax = axes[1, 0]
ax.plot(
    scale_summary["delta_x"],
    scale_summary["median_epsilon2"],
    marker="o",
)
ax.set_xlabel(f"{x_col_used} window width, Δx")
ax.set_ylabel(r"Median $\epsilon^2$")
ax.set_title("C. Kruskal effect size")
ax.grid(True, alpha=0.3)

# -----------------------------
# D. Spearman with sign
# -----------------------------
ax = axes[1, 1]
x = scale_summary["delta_x"]
y = scale_summary["median_spearman"]
lo = scale_summary["q25_spearman"]
hi = scale_summary["q75_spearman"]

ax.plot(x, y, marker="o", label="median signed ρ")
ax.fill_between(x, lo, hi, alpha=0.2, label="IQR")
ax.axhline(0, linestyle="--", alpha=0.7)
ax.set_xlabel(f"{x_col_used} window width, Δx")
ax.set_ylabel("Spearman ρ")
ax.set_title("D. Ordinal association with sign")
ax.grid(True, alpha=0.3)
ax.legend()

fig.suptitle(
    f"Scale dependence of {x_col_used}-comfort structure ({ANALYSIS_MODE})",
    fontsize=15,
)

plt.show()

In [ ]:
# ============================================================
# SCALE-SPACE HEATMAP
# ============================================================

center_bin_width = DELTA_STEP * 2
min_cell_support = 1
clip_p_min = 1e-12

# Optional reference lines
# For temperature, put around 29 if relevant
transition_lines = []

if "T" in x_col_used:
    transition_lines = [29]
elif "HDX" in x_col_used:
    transition_lines = [32, 37.5]

heat = scale_offsets[scale_offsets["valid_test"]].copy()

heat["minus_log10_p"] = -np.log10(
    heat["kw_p"].clip(lower=clip_p_min)
)

heat["center_bin"] = (
    np.round(heat["center"] / center_bin_width) * center_bin_width
)

pivot_value = heat.pivot_table(
    index="delta_x",
    columns="center_bin",
    values="minus_log10_p",
    aggfunc="median",
)

pivot_support = heat.pivot_table(
    index="delta_x",
    columns="center_bin",
    values="minus_log10_p",
    aggfunc="count",
)

pivot_value = pivot_value.sort_index().sort_index(axis=1)
pivot_support = pivot_support.reindex_like(pivot_value)

pivot_masked = pivot_value.mask(pivot_support < min_cell_support)

# Edges for pcolormesh
x_centers = pivot_masked.columns.to_numpy(dtype=float)
y_centers = pivot_masked.index.to_numpy(dtype=float)

x_edges = np.r_[
    x_centers[0] - center_bin_width / 2,
    x_centers + center_bin_width / 2,
]

if len(y_centers) > 1:
    y_mid = 0.5 * (y_centers[:-1] + y_centers[1:])
    y_edges = np.r_[
        y_centers[0] - (y_mid[0] - y_centers[0]),
        y_mid,
        y_centers[-1] + (y_centers[-1] - y_mid[-1]),
    ]
else:
    dy = 0.5
    y_edges = np.array([y_centers[0] - dy, y_centers[0] + dy])

Z = np.ma.masked_invalid(pivot_masked.to_numpy())

# Observed coverage
x_obs = df_scale[x_col_used].dropna().to_numpy()

bin_edges = np.arange(
    x_edges.min(),
    x_edges.max() + center_bin_width,
    center_bin_width,
)

counts, edges = np.histogram(x_obs, bins=bin_edges)
count_centers = 0.5 * (edges[:-1] + edges[1:])

xlim_low = np.nanmin(x_obs) - 0.5
xlim_high = np.nanmax(x_obs) + 0.5

# Plot
fig = plt.figure(figsize=(13, 8))

gs = gridspec.GridSpec(
    2, 2,
    width_ratios=[20, 1],
    height_ratios=[1.2, 4],
    wspace=0.12,
    hspace=0.18,
)

ax_top = fig.add_subplot(gs[0, 0])
ax = fig.add_subplot(gs[1, 0], sharex=ax_top)
cax = fig.add_subplot(gs[1, 1])

ax_empty = fig.add_subplot(gs[0, 1])
ax_empty.axis("off")

# Top histogram
ax_top.bar(
    count_centers,
    counts,
    width=center_bin_width * 0.9,
    align="center",
    alpha=0.75,
)

ax_top.set_ylabel("Votes\nper bin")
ax_top.set_title(f"Observed {x_col_used} coverage", fontsize=12)
ax_top.grid(True, axis="y", alpha=0.3)
ax_top.tick_params(axis="x", labelbottom=True)

for xc in transition_lines:
    ax_top.axvline(
        xc,
        linestyle="--",
        color="black",
        alpha=0.6,
        linewidth=1.2,
    )

# Heatmap
cmap = plt.cm.viridis.copy()
cmap.set_bad(color="white")

mesh = ax.pcolormesh(
    x_edges,
    y_edges,
    Z,
    cmap=cmap,
    shading="auto",
)

cbar = fig.colorbar(mesh, cax=cax)
cbar.set_label("Median -log10(Kruskal p)")

for xc in transition_lines:
    ax.axvline(
        xc,
        linestyle="--",
        color="white",
        alpha=0.85,
        linewidth=1.5,
    )

    ax.text(
        xc,
        y_edges[0] + 0.2,
        str(xc),
        color="white",
        rotation=90,
        va="bottom",
        ha="right",
        fontsize=9,
    )

ax.set_xlabel(f"{x_col_used} window center")
ax.set_ylabel(f"{x_col_used} window width, Δx")
ax.set_title(
    f"Scale-space map of comfort-class statistical detectability ({ANALYSIS_MODE})",
    fontsize=12,
)

ax.set_xlim(xlim_low, xlim_high)
ax.set_ylim(y_edges[0], y_edges[-1])

fig.suptitle(
    f"Observed {x_col_used} coverage and scale-space statistical detectability",
    fontsize=15,
    y=0.98,
)

plt.show()

# utlitzem les categories binaries? 

In [ ]:
comfort_to_side3 = {
    "Very comfortable": "1. Comfortable",
    "Comfortable": "1. Comfortable",
    "Slightly comfortable": "1. Comfortable",

    "Neutral": "2. Neutral",

    "Slightly uncomfortable": "3. Discomfort",
    "Uncomfortable": "3. Discomfort",
    "Very uncomfortable": "3. Discomfort",
}

side3_order = [
    "1. Comfortable",
    "2. Neutral",
    "3. Discomfort",
]

side3_score = {
    "1. Comfortable": 1,
    "2. Neutral": 2,
    "3. Discomfort": 3,
}

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# ============================================================
# CONFIG
# ============================================================

#X_COL = "<HDX-HDX_fixed+<HDX>>"
X_COL = "<T-T_fixed+<T>>"
# X_COL = "T_K"

COMFORT_COL = "thermal_comfort"

ANALYSIS_MODE = "comfort_neutral_discomfort"

# If using T_K and it is Kelvin
AUTO_CONVERT_KELVIN = True

MIN_N = 30
MIN_CLASS_N = 3
N_OFFSETS_PER_DELTA = 25

# For HDX
# DELTA_MIN = 1.0
# DELTA_STEP = 0.5
# DELTA_MAX_CAP = 20

#For temperature, use instead:
DELTA_MIN = 0.5
DELTA_STEP = 0.25
DELTA_MAX_CAP = 12

# ============================================================
# PREPARE DATA
# ============================================================

df_scale = df_votes.copy()

df_scale[X_COL] = pd.to_numeric(df_scale[X_COL], errors="coerce")

x_col_used = X_COL

if AUTO_CONVERT_KELVIN and X_COL == "T_K":
    if df_scale[X_COL].median() > 100:
        df_scale["T_C"] = df_scale[X_COL] - 273.15
        x_col_used = "T_C"

df_scale["comfort_side3"] = df_scale[COMFORT_COL].map(comfort_to_side3)
df_scale["comfort_side3_score"] = df_scale["comfort_side3"].map(side3_score)

df_scale = df_scale.dropna(
    subset=[x_col_used, "comfort_side3", "comfort_side3_score"]
).copy()

df_scale = df_scale[df_scale["comfort_side3"].isin(side3_order)].copy()

print("Environmental variable:", x_col_used)
print("Analysis mode:", ANALYSIS_MODE)
print("N:", len(df_scale))

print("\nGroup counts:")
print(df_scale["comfort_side3"].value_counts().reindex(side3_order))

print("\nEnvironmental summary:")
print(df_scale[x_col_used].describe().round(3))

In [ ]:
x_range = df_scale[x_col_used].max() - df_scale[x_col_used].min()

delta_values = np.arange(
    DELTA_MIN,
    min(0.95 * x_range, DELTA_MAX_CAP) + 0.001,
    DELTA_STEP,
)

all_results = []

for delta in delta_values:
    offsets = np.linspace(0, delta, N_OFFSETS_PER_DELTA, endpoint=False)

    for offset in offsets:
        out = analyze_one_tiling_scale(
            df=df_scale,
            x_col=x_col_used,
            label_col="comfort_side3",
            score_col="comfort_side3_score",
            class_order=side3_order,
            delta_x=delta,
            offset=offset,
            min_n=MIN_N,
            min_class_n=MIN_CLASS_N,
        )

        all_results.append(out)

scale_offsets_side3 = pd.concat(all_results, ignore_index=True)

print(scale_offsets_side3.shape)
scale_offsets_side3.head()

In [ ]:
valid_side3 = scale_offsets_side3[
    scale_offsets_side3["valid_test"]
].copy()

valid_side3["minus_log10_p"] = -np.log10(
    valid_side3["kw_p"].clip(lower=1e-12)
)

scale_summary_side3 = (
    valid_side3
    .groupby("delta_x")
    .agg(
        n_windows=("kw_p", "count"),

        frac_kw_significant=("kw_p", lambda x: np.mean(x < 0.05)),

        median_kw_p=("kw_p", "median"),
        q25_kw_p=("kw_p", lambda x: np.nanquantile(x, 0.25)),
        q75_kw_p=("kw_p", lambda x: np.nanquantile(x, 0.75)),

        median_minus_log10_p=("minus_log10_p", "median"),
        q25_minus_log10_p=("minus_log10_p", lambda x: np.nanquantile(x, 0.25)),
        q75_minus_log10_p=("minus_log10_p", lambda x: np.nanquantile(x, 0.75)),

        median_kw_H=("kw_H", "median"),
        median_epsilon2=("epsilon2", "median"),

        median_spearman=("spearman_rho", "median"),
        mean_spearman=("spearman_rho", "mean"),
        q25_spearman=("spearman_rho", lambda x: np.nanquantile(x, 0.25)),
        q75_spearman=("spearman_rho", lambda x: np.nanquantile(x, 0.75)),

        frac_positive_spearman=("spearman_rho", lambda x: np.mean(x > 0)),
        median_abs_spearman=("spearman_rho", lambda x: np.nanmedian(np.abs(x))),
    )
    .reset_index()
)

scale_summary_side3.round(4)

In [ ]:
fig, axes = plt.subplots(
    2, 2,
    figsize=(13, 9),
    constrained_layout=True,
)

# A. Fraction significant
ax = axes[0, 0]
ax.plot(
    scale_summary_side3["delta_x"],
    scale_summary_side3["frac_kw_significant"],
    marker="o",
)
ax.axhline(0.3, linestyle="--", alpha=0.7, label="30%")
ax.axhline(0.5, linestyle="--", alpha=0.7, label="50%")
ax.set_xlabel(f"{x_col_used} window width, Δx")
ax.set_ylabel("Fraction Kruskal p < 0.05")
ax.set_title("A. Statistical detectability")
ax.grid(True, alpha=0.3)
ax.legend()

# B. -log10 p
ax = axes[0, 1]
x = scale_summary_side3["delta_x"]
y = scale_summary_side3["median_minus_log10_p"]
lo = scale_summary_side3["q25_minus_log10_p"]
hi = scale_summary_side3["q75_minus_log10_p"]

ax.plot(x, y, marker="o", label="median")
ax.fill_between(x, lo, hi, alpha=0.2, label="IQR")
ax.axhline(-np.log10(0.05), linestyle="--", alpha=0.7, label="p = 0.05")
ax.set_xlabel(f"{x_col_used} window width, Δx")
ax.set_ylabel("Median -log10(Kruskal p)")
ax.set_title("B. Strength of Kruskal evidence")
ax.grid(True, alpha=0.3)
ax.legend()

# C. epsilon2
ax = axes[1, 0]
ax.plot(
    scale_summary_side3["delta_x"],
    scale_summary_side3["median_epsilon2"],
    marker="o",
)
ax.set_xlabel(f"{x_col_used} window width, Δx")
ax.set_ylabel(r"Median $\epsilon^2$")
ax.set_title("C. Kruskal effect size")
ax.grid(True, alpha=0.3)

# D. signed Spearman
ax = axes[1, 1]
x = scale_summary_side3["delta_x"]
y = scale_summary_side3["median_spearman"]
lo = scale_summary_side3["q25_spearman"]
hi = scale_summary_side3["q75_spearman"]

ax.plot(x, y, marker="o", label="median signed ρ")
ax.fill_between(x, lo, hi, alpha=0.2, label="IQR")
ax.axhline(0, linestyle="--", alpha=0.7)
ax.set_xlabel(f"{x_col_used} window width, Δx")
ax.set_ylabel("Spearman ρ")
ax.set_title("D. Ordinal association with sign")
ax.grid(True, alpha=0.3)
ax.legend()

fig.suptitle(
    f"Scale dependence of {x_col_used}-comfort structure "
    f"(comfortable / neutral / discomfort)",
    fontsize=15,
)

plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), constrained_layout=True)

# A. Kruskal fraction
ax = axes[0]
ax.plot(
    scale_summary_T["delta_x"],
    scale_summary_T["frac_kw_significant"],
    marker="o",
    label="Previous 3 groups",
)
# ax.plot(
#     scale_summary_side3["delta_x"],
#     scale_summary_side3["frac_kw_significant"],
#     marker="o",
#     label="Comfort / Neutral / Discomfort",
# )

ax.set_xlabel("Window width, Δx")
ax.set_ylabel("Fraction p < 0.05")
ax.set_title("A. Statistical detectability")
ax.grid(True, alpha=0.3)
ax.legend()

# B. epsilon2
ax = axes[1]
ax.plot(
    scale_summary_T["delta_x"],
    scale_summary_T["median_epsilon2"],
    marker="o",
    label="Previous 3 groups",
)
# ax.plot(
#     scale_summary_side3["delta_x"],
#     scale_summary_side3["median_epsilon2"],
#     marker="o",
#     label="Comfort / Neutral / Discomfort",
# )
ax.set_xlabel("Window width, Δx")
ax.set_ylabel(r"Median $\epsilon^2$")
ax.set_title("B. Effect size")
ax.grid(True, alpha=0.3)
ax.legend()

# C. signed Spearman
ax = axes[2]
ax.plot(
    scale_summary_T["delta_x"],
    scale_summary_T["median_spearman"],
    marker="o",
    label="Previous 3 groups",
)
# ax.plot(
#     scale_summary_side3["delta_x"],
#     scale_summary_side3["median_spearman"],
#     marker="o",
#     label="Comfort / Neutral / Discomfort",
# )
ax.axhline(0, linestyle="--", alpha=0.6)
ax.set_xlabel("Window width, Δx")
ax.set_ylabel("Median signed Spearman ρ")
ax.set_title("C. Ordinal association")
ax.grid(True, alpha=0.3)
ax.legend()

fig.suptitle(
    f"Effect of comfort grouping on scale dependence ({x_col_used})",
    fontsize=15,
)

plt.show()

# i amb 7 categories

In [ ]:
# ============================================================
# 7 ORIGINAL COMFORT CATEGORIES
# ============================================================

comfort_order_7 = [
    "Very comfortable",
    "Comfortable",
    "Slightly comfortable",
    "Neutral",
    "Slightly uncomfortable",
    "Uncomfortable",
    "Very uncomfortable",
]

comfort_score_7 = {
    "Very comfortable": 1,
    "Comfortable": 2,
    "Slightly comfortable": 3,
    "Neutral": 4,
    "Slightly uncomfortable": 5,
    "Uncomfortable": 6,
    "Very uncomfortable": 7,
}

df_scale_7 = df_votes.copy()

df_scale_7[x_col_used] = pd.to_numeric(df_scale_7[x_col_used], errors="coerce")

df_scale_7["comfort_7"] = df_scale_7[COMFORT_COL]
df_scale_7["comfort_7_score"] = df_scale_7["comfort_7"].map(comfort_score_7)

df_scale_7 = df_scale_7.dropna(
    subset=[x_col_used, "comfort_7", "comfort_7_score"]
).copy()

df_scale_7 = df_scale_7[
    df_scale_7["comfort_7"].isin(comfort_order_7)
].copy()

print("N:", len(df_scale_7))
print(df_scale_7["comfort_7"].value_counts().reindex(comfort_order_7))

In [ ]:
# ============================================================
# RUN SCALE ANALYSIS — 7 CATEGORIES
# ============================================================

# Use the same delta values as in the previous analyses
delta_values_7 = scale_summary["delta_x"].values

all_results_7 = []

for delta in delta_values_7:
    offsets = np.linspace(0, delta, N_OFFSETS_PER_DELTA, endpoint=False)

    for offset in offsets:
        out = analyze_one_tiling_scale(
            df=df_scale_7,
            x_col=x_col_used,
            label_col="comfort_7",
            score_col="comfort_7_score",
            class_order=comfort_order_7,
            delta_x=delta,
            offset=offset,
            min_n=MIN_N,
            min_class_n=MIN_CLASS_N,
        )

        all_results_7.append(out)

scale_offsets_7 = pd.concat(all_results_7, ignore_index=True)

print(scale_offsets_7.shape)
scale_offsets_7.head()

In [ ]:
# ============================================================
# SUMMARY — 7 CATEGORIES
# ============================================================

valid_7 = scale_offsets_7[scale_offsets_7["valid_test"]].copy()

valid_7["minus_log10_p"] = -np.log10(
    valid_7["kw_p"].clip(lower=1e-12)
)

scale_summary_7 = (
    valid_7
    .groupby("delta_x")
    .agg(
        n_windows=("kw_p", "count"),

        frac_kw_significant=("kw_p", lambda x: np.mean(x < 0.05)),

        median_kw_p=("kw_p", "median"),
        q25_kw_p=("kw_p", lambda x: np.nanquantile(x, 0.25)),
        q75_kw_p=("kw_p", lambda x: np.nanquantile(x, 0.75)),

        median_minus_log10_p=("minus_log10_p", "median"),
        q25_minus_log10_p=("minus_log10_p", lambda x: np.nanquantile(x, 0.25)),
        q75_minus_log10_p=("minus_log10_p", lambda x: np.nanquantile(x, 0.75)),

        median_kw_H=("kw_H", "median"),
        median_epsilon2=("epsilon2", "median"),

        median_spearman=("spearman_rho", "median"),
        mean_spearman=("spearman_rho", "mean"),
        q25_spearman=("spearman_rho", lambda x: np.nanquantile(x, 0.25)),
        q75_spearman=("spearman_rho", lambda x: np.nanquantile(x, 0.75)),

        frac_positive_spearman=("spearman_rho", lambda x: np.mean(x > 0)),
        median_abs_spearman=("spearman_rho", lambda x: np.nanmedian(np.abs(x))),
    )
    .reset_index()
)

scale_summary_7.round(4)

In [ ]:
# ============================================================
# COMPARISON PLOT
# Previous 3 groups vs Comfort/Neutral/Discomfort vs 7 categories
# ============================================================

fig, axes = plt.subplots(
    1, 3,
    figsize=(17, 4.8),
    constrained_layout=True,
)

# ------------------------------------------------------------
# A. Statistical detectability
# ------------------------------------------------------------

ax = axes[0]

ax.plot(
    scale_summary["delta_x"],
    scale_summary["frac_kw_significant"],
    marker="o",
    label="Previous 3 groups",
)

ax.plot(
    scale_summary_side3["delta_x"],
    scale_summary_side3["frac_kw_significant"],
    marker="o",
    label="Comfort / Neutral / Discomfort",
)

ax.plot(
    scale_summary_7["delta_x"],
    scale_summary_7["frac_kw_significant"],
    marker="o",
    label="7 original categories",
)

ax.axhline(0.5, linestyle="--", alpha=0.6)
ax.set_xlabel("Window width, Δx")
ax.set_ylabel("Fraction Kruskal p < 0.05")
ax.set_title("A. Statistical detectability")
ax.grid(True, alpha=0.3)
ax.legend()


# ------------------------------------------------------------
# B. Effect size
# ------------------------------------------------------------

ax = axes[1]

ax.plot(
    scale_summary["delta_x"],
    scale_summary["median_epsilon2"],
    marker="o",
    label="Previous 3 groups",
)

ax.plot(
    scale_summary_side3["delta_x"],
    scale_summary_side3["median_epsilon2"],
    marker="o",
    label="Comfort / Neutral / Discomfort",
)

ax.plot(
    scale_summary_7["delta_x"],
    scale_summary_7["median_epsilon2"],
    marker="o",
    label="7 original categories",
)

ax.set_xlabel("Window width, Δx")
ax.set_ylabel(r"Median $\epsilon^2$")
ax.set_title("B. Kruskal effect size")
ax.grid(True, alpha=0.3)
ax.legend()


# ------------------------------------------------------------
# C. Ordinal association
# ------------------------------------------------------------

ax = axes[2]

ax.plot(
    scale_summary["delta_x"],
    scale_summary["median_spearman"],
    marker="o",
    label="Previous 3 groups",
)

ax.plot(
    scale_summary_side3["delta_x"],
    scale_summary_side3["median_spearman"],
    marker="o",
    label="Comfort / Neutral / Discomfort",
)

ax.plot(
    scale_summary_7["delta_x"],
    scale_summary_7["median_spearman"],
    marker="o",
    label="7 original categories",
)

ax.axhline(0, linestyle="--", alpha=0.6)
ax.set_xlabel("Window width, Δx")
ax.set_ylabel("Median signed Spearman ρ")
ax.set_title("C. Ordinal association")
ax.grid(True, alpha=0.3)
ax.legend()

fig.suptitle(
    f"Effect of comfort representation on scale dependence ({x_col_used})",
    fontsize=15,
    y=1.05,
)

plt.show()

scale_summary_T = scale_summary.copy()

In [ ]:
usable_classes_summary_7 = (
    scale_offsets_7[scale_offsets_7["valid_test"]]
    .groupby("delta_x")
    .agg(
        median_usable_classes=("n_usable_classes", "median"),
        mean_usable_classes=("n_usable_classes", "mean"),
        q25_usable_classes=("n_usable_classes", lambda x: np.nanquantile(x, 0.25)),
        q75_usable_classes=("n_usable_classes", lambda x: np.nanquantile(x, 0.75)),
    )
    .reset_index()
)

usable_classes_summary_7.round(3)

# mirem si el problema és que neutral no funciona tan bé com un estat intermig i l'ajuntem dins del binari de comfort

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# CHOOSE PREDICTOR
# ============================================================

# posa aquí la variable que vulguis analitzar
# x_col = "<HDX-HDX_fixed+<HDX>>"
x_col = "<T-T_fixed+<T>>"

comfort_col = "thermal_comfort"

# ============================================================
# LOAD / PREP DATA
# ============================================================

df_scale = df_votes.copy()
df_scale[x_col] = pd.to_numeric(df_scale[x_col], errors="coerce")

# ============================================================
# 2-GROUP COMFORT REPRESENTATION
# ============================================================

def comfort_2group(cat):
    if cat in [
        "Very comfortable",
        "Comfortable",
        "Slightly comfortable",
        "Neutral",
    ]:
        return "1. Comfort / neutral"
    elif cat in [
        "Slightly uncomfortable",
        "Uncomfortable",
        "Very uncomfortable",
    ]:
        return "2. Discomfort"
    return np.nan

df_scale["comfort_2group"] = df_scale[comfort_col].map(comfort_2group)

# ordinal score
score_map_2 = {
    "1. Comfort / neutral": 1,
    "2. Discomfort": 2,
}
df_scale["comfort_2group_score"] = df_scale["comfort_2group"].map(score_map_2)

# keep valid rows
class_order_2 = [
    "1. Comfort / neutral",
    "2. Discomfort",
]

df_scale = df_scale[
    df_scale["comfort_2group"].isin(class_order_2)
].dropna(subset=[x_col, "comfort_2group", "comfort_2group_score"]).copy()

print(df_scale["comfort_2group"].value_counts())

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

def analyze_one_tiling(
    df,
    x_col,
    label_col,
    score_col,
    class_order,
    delta_hdx,
    offset=0.0,
    min_n=30,
    min_class_n=3,
):
    """
    Creates non-overlapping bins of width delta_hdx.
    The offset shifts the bin origin, so we can test many possible tilings.
    """

    x_min = df[x_col].min()
    x_max = df[x_col].max()

    start = x_min - delta_hdx + offset
    edges = np.arange(start, x_max + 2 * delta_hdx, delta_hdx)

    d = df.copy()
    d["window_bin"] = pd.cut(
        d[x_col],
        bins=edges,
        include_lowest=True,
        right=False,
    )

    rows = []

    for window, sub in d.groupby("window_bin", observed=False):
        if sub.empty:
            continue

        low = window.left
        high = window.right
        center = 0.5 * (low + high)
        n = len(sub)

        counts = (
            sub[label_col]
            .value_counts()
            .reindex(class_order)
            .fillna(0)
        )

        present_classes = counts[counts > 0].index.tolist()
        usable_classes = counts[counts >= min_class_n].index.tolist()

        # entropy of response distribution
        p_counts = counts[counts > 0] / counts.sum()
        entropy = -(p_counts * np.log(p_counts)).sum() if len(p_counts) > 0 else np.nan

        valid_test = (n >= min_n) and (len(usable_classes) >= 2)

        H = np.nan
        p_kw = np.nan
        epsilon2 = np.nan
        rho = np.nan
        p_spear = np.nan
        mean_spread = np.nan
        spread_over_iqr = np.nan
        spread_over_sd = np.nan

        if valid_test:
            groups = [
                sub.loc[sub[label_col] == c, x_col].dropna().values
                for c in usable_classes
            ]

            H, p_kw = stats.kruskal(*groups)

            N = sum(len(g) for g in groups)
            k = len(groups)
            epsilon2 = (H - k + 1) / (N - k) if N > k else np.nan

            rho, p_spear = stats.spearmanr(
                sub[x_col],
                sub[score_col],
                nan_policy="omit",
            )

            means = (
                sub.groupby(label_col, observed=False)[x_col]
                .mean()
                .reindex(class_order)
            )

            means = means.loc[
                [c for c in means.index if counts.get(c, 0) >= min_class_n]
            ].dropna()

            if len(means) >= 2:
                mean_spread = means.max() - means.min()

                x_iqr = sub[x_col].quantile(0.75) - sub[x_col].quantile(0.25)
                x_sd = sub[x_col].std()

                spread_over_iqr = mean_spread / x_iqr if x_iqr > 0 else np.nan
                spread_over_sd = mean_spread / x_sd if x_sd > 0 else np.nan

        rows.append({
            "delta_x": delta_hdx,
            "offset": offset,
            "window": str(window),
            "low": low,
            "high": high,
            "center": center,
            "n": n,
            "n_present_classes": len(present_classes),
            "n_usable_classes": len(usable_classes),
            "entropy": entropy,
            "valid_test": valid_test,
            "kw_H": H,
            "kw_p": p_kw,
            "epsilon2": epsilon2,
            "spearman_rho": rho,
            "spearman_p": p_spear,
            "mean_spread": mean_spread,
            "spread_over_iqr": spread_over_iqr,
            "spread_over_sd": spread_over_sd,
        })

    return pd.DataFrame(rows)

In [ ]:
def run_scale_analysis(
    df,
    x_col,
    label_col,
    score_col,
    class_order,
    delta_values,
    n_offsets=25,
    min_n=30,
    min_class_n=3,
):
    all_rows = []

    x_min = df[x_col].min()

    for delta_x in delta_values:
        offsets = np.linspace(0, delta_x, n_offsets, endpoint=False)

        for offset in offsets:
            res = analyze_one_tiling(
                df=df,
                x_col=x_col,
                label_col=label_col,
                score_col=score_col,
                class_order=class_order,
                delta_hdx=delta_x,
                offset=offset,
                min_n=min_n,
                min_class_n=min_class_n,
            )
            all_rows.append(res)

    scale_offsets = pd.concat(all_rows, ignore_index=True)

    return scale_offsets


delta_values = np.arange(0.5, 11.6, 0.25)

scale_offsets_2groups = run_scale_analysis(
    df=df_scale,
    x_col=x_col,
    label_col="comfort_2group",
    score_col="comfort_2group_score",
    class_order=class_order_2,
    delta_values=delta_values,
    n_offsets=25,
    min_n=30,
    min_class_n=3,
)

print(scale_offsets_2groups.shape)
scale_offsets_2groups.head()

In [ ]:
scale_summary_2groups = (
    scale_offsets_2groups
    .loc[scale_offsets_2groups["valid_test"]]
    .groupby("delta_x", as_index=False)
    .agg(
        n_windows=("kw_p", "size"),
        frac_kw_significant=("kw_p", lambda s: np.mean(s < 0.05)),
        mean_kw_p=("kw_p", "mean"),
        median_kw_p=("kw_p", "median"),
        q25_kw_p=("kw_p", lambda s: s.quantile(0.25)),
        q75_kw_p=("kw_p", lambda s: s.quantile(0.75)),

        mean_minus_log10_p=("kw_p", lambda s: np.mean(-np.log10(np.clip(s, 1e-12, None)))),
        median_minus_log10_p=("kw_p", lambda s: np.median(-np.log10(np.clip(s, 1e-12, None)))),
        q25_minus_log10_p=("kw_p", lambda s: np.quantile(-np.log10(np.clip(s, 1e-12, None)), 0.25)),
        q75_minus_log10_p=("kw_p", lambda s: np.quantile(-np.log10(np.clip(s, 1e-12, None)), 0.75)),

        median_kw_H=("kw_H", "median"),
        mean_kw_H=("kw_H", "mean"),

        median_epsilon2=("epsilon2", "median"),
        mean_epsilon2=("epsilon2", "mean"),

        median_spearman=("spearman_rho", "median"),
        mean_spearman=("spearman_rho", "mean"),
        q25_spearman=("spearman_rho", lambda s: s.quantile(0.25)),
        q75_spearman=("spearman_rho", lambda s: s.quantile(0.75)),

        median_abs_spearman=("spearman_rho", lambda s: np.median(np.abs(s))),
        frac_positive_spearman=("spearman_rho", lambda s: np.mean(s > 0)),
    )
)

scale_summary_2groups.head()

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# A. detectability
axes[0].plot(
    scale_summary_2groups["delta_x"],
    scale_summary_2groups["frac_kw_significant"],
    marker="o",
    label="2 groups: Comfort+Neutral / Discomfort",
)
axes[0].axhline(0.5, linestyle="--", alpha=0.7)
axes[0].set_xlabel(f"{x_col} window width, Δx")
axes[0].set_ylabel("Fraction Kruskal p < 0.05")
axes[0].set_title("A. Statistical detectability")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# B. effect size
axes[1].plot(
    scale_summary_2groups["delta_x"],
    scale_summary_2groups["median_epsilon2"],
    marker="o",
    label="2 groups",
)
axes[1].set_xlabel(f"{x_col} window width, Δx")
axes[1].set_ylabel("Median epsilon²")
axes[1].set_title("B. Kruskal effect size")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

# C. ordinal association
axes[2].plot(
    scale_summary_2groups["delta_x"],
    scale_summary_2groups["median_spearman"],
    marker="o",
    label="2 groups",
)
axes[2].axhline(0, linestyle="--", alpha=0.7)
axes[2].set_xlabel(f"{x_col} window width, Δx")
axes[2].set_ylabel("Median signed Spearman rho")
axes[2].set_title("C. Ordinal association")
axes[2].grid(True, alpha=0.3)
axes[2].legend()

plt.suptitle(f"Scale dependence with 2 comfort groups ({x_col})", y=1.02, fontsize=15)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --------------------------------------------------
# A. Detectability
# --------------------------------------------------
axes[0].plot(scale_summary["delta_x"], scale_summary["frac_kw_significant"], marker="o", label="Previous 3 groups")
axes[0].plot(scale_summary_side3["delta_x"], scale_summary_side3["frac_kw_significant"], marker="o", label="Comfort / Neutral / Discomfort")
axes[0].plot(scale_summary_7["delta_x"], scale_summary_7["frac_kw_significant"], marker="o", label="7 original categories")
axes[0].plot(scale_summary_2groups["delta_x"], scale_summary_2groups["frac_kw_significant"], marker="o", label="2 groups")
axes[0].axhline(0.5, linestyle="--", alpha=0.7)
axes[0].set_xlabel("Window width, Δx")
axes[0].set_ylabel("Fraction Kruskal p < 0.05")
axes[0].set_title("A. Statistical detectability")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# --------------------------------------------------
# B. Effect size
# --------------------------------------------------
axes[1].plot(scale_summary["delta_x"], scale_summary["median_epsilon2"], marker="o", label="Previous 3 groups")
axes[1].plot(scale_summary_side3["delta_x"], scale_summary_side3["median_epsilon2"], marker="o", label="Comfort / Neutral / Discomfort")
axes[1].plot(scale_summary_7["delta_x"], scale_summary_7["median_epsilon2"], marker="o", label="7 original categories")
axes[1].plot(scale_summary_2groups["delta_x"], scale_summary_2groups["median_epsilon2"], marker="o", label="2 groups")
axes[1].set_xlabel("Window width, Δx")
axes[1].set_ylabel("Median epsilon²")
axes[1].set_title("B. Kruskal effect size")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

# --------------------------------------------------
# C. Spearman
# --------------------------------------------------
axes[2].plot(scale_summary["delta_x"], scale_summary["median_spearman"], marker="o", label="Previous 3 groups")
axes[2].plot(scale_summary_side3["delta_x"], scale_summary_side3["median_spearman"], marker="o", label="Comfort / Neutral / Discomfort")
axes[2].plot(scale_summary_7["delta_x"], scale_summary_7["median_spearman"], marker="o", label="7 original categories")
axes[2].plot(scale_summary_2groups["delta_x"], scale_summary_2groups["median_spearman"], marker="o", label="2 groups")
axes[2].axhline(0, linestyle="--", alpha=0.7)
axes[2].set_xlabel("Window width, Δx")
axes[2].set_ylabel("Median signed Spearman rho")
axes[2].set_title("C. Ordinal association")
axes[2].grid(True, alpha=0.3)
axes[2].legend()

plt.suptitle(f"Effect of comfort representation on scale dependence ({x_col})", y=1.03, fontsize=15)
plt.tight_layout()
plt.show()

# ara anem a incorporar els errors a la comparació entre

3 Original

3 Binari

7 estats

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def get_delta_col(df):
    if "delta_x" in df.columns:
        return "delta_x"
    elif "delta_hdx" in df.columns:
        return "delta_hdx"
    else:
        raise ValueError("No delta_x or delta_hdx column found.")


def compute_metrics(d):
    kw_p = d["kw_p"].dropna().values
    eps = d["epsilon2"].dropna().values
    rho = d["spearman_rho"].dropna().values

    out = {
        "frac_kw_significant": np.mean(kw_p < 0.05) if len(kw_p) else np.nan,
        "median_minus_log10_p": np.nanmedian(-np.log10(np.clip(kw_p, 1e-12, None))) if len(kw_p) else np.nan,
        "median_epsilon2": np.nanmedian(eps) if len(eps) else np.nan,
        "median_spearman": np.nanmedian(rho) if len(rho) else np.nan,
    }

    return out


def bootstrap_offset_summary(
    scale_df,
    label,
    n_boot=1000,
    ci=(2.5, 97.5),
    random_state=123,
):
    rng = np.random.default_rng(random_state)

    d = scale_df[scale_df["valid_test"]].copy()
    delta_col = get_delta_col(d)

    rows = []

    for delta, d_delta in d.groupby(delta_col):
        # central line: exactly the same logic as your original plot
        central = compute_metrics(d_delta)

        # group valid windows by offset
        offset_groups = [
            g.copy()
            for _, g in d_delta.groupby("offset")
            if len(g) > 0
        ]

        n_offsets = len(offset_groups)

        boot_vals = {
            "frac_kw_significant": [],
            "median_minus_log10_p": [],
            "median_epsilon2": [],
            "median_spearman": [],
        }

        if n_offsets >= 2:
            for _ in range(n_boot):
                sampled_ids = rng.integers(0, n_offsets, size=n_offsets)
                sampled = pd.concat(
                    [offset_groups[i] for i in sampled_ids],
                    ignore_index=True,
                )

                m = compute_metrics(sampled)

                for key in boot_vals:
                    boot_vals[key].append(m[key])

        row = {
            "delta": delta,
            "representation": label,
            "n_windows": len(d_delta),
            "n_offsets": n_offsets,
        }

        for key, value in central.items():
            row[f"{key}_center"] = value

            if n_offsets >= 2:
                row[f"{key}_low"] = np.nanpercentile(boot_vals[key], ci[0])
                row[f"{key}_high"] = np.nanpercentile(boot_vals[key], ci[1])
            else:
                row[f"{key}_low"] = np.nan
                row[f"{key}_high"] = np.nan

        rows.append(row)

    return pd.DataFrame(rows).sort_values("delta")

In [ ]:
summary_prev3 = bootstrap_offset_summary(
    scale_offsets,
    "Previous 3 groups",
    n_boot=1000,
)

summary_7 = bootstrap_offset_summary(
    scale_offsets_7,
    "7 original categories",
    n_boot=1000,
)

summary_side3 = bootstrap_offset_summary(
    scale_offsets_side3,
    "Comfort / Neutral / Uncomfortable",
    n_boot=1000,
)

summaries = [
    ("Previous 3 groups", summary_prev3),
    ("7 original categories", summary_7),
    ("Comfort / Neutral / Uncomfortable", summary_side3),
]

In [ ]:
fig, axes = plt.subplots(
    1, 3,
    figsize=(18, 5),
    constrained_layout=True,
)

# --------------------------------------------------
# A. Statistical detectability
# --------------------------------------------------
ax = axes[0]

for label, s in summaries:
    ax.plot(
        s["delta"],
        s["frac_kw_significant_center"],
        marker="o",
        label=label,
    )
    ax.fill_between(
        s["delta"],
        s["frac_kw_significant_low"],
        s["frac_kw_significant_high"],
        alpha=0.16,
    )


ax.axhline(0.5, linestyle="--", alpha=0.6, label="50%")

ax.set_xlabel("Window width, Δx")
ax.set_ylabel("Fraction Kruskal p < 0.05")
ax.set_title("A. Statistical detectability")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8)


# --------------------------------------------------
# B. Kruskal effect size
# --------------------------------------------------
ax = axes[1]

for label, s in summaries:
    ax.plot(
        s["delta"],
        s["median_epsilon2_center"],
        marker="o",
        label=label,
    )
    ax.fill_between(
        s["delta"],
        s["median_epsilon2_low"],
        s["median_epsilon2_high"],
        alpha=0.16,
    )

ax.set_xlabel("Window width, Δx")
ax.set_ylabel(r"Median $\epsilon^2$")
ax.set_title("B. Kruskal effect size")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8)


# --------------------------------------------------
# C. Ordinal association
# --------------------------------------------------
ax = axes[2]

for label, s in summaries:
    ax.plot(
        s["delta"],
        s["median_spearman_center"],
        marker="o",
        label=label,
    )
    ax.fill_between(
        s["delta"],
        s["median_spearman_low"],
        s["median_spearman_high"],
        alpha=0.16,
    )

ax.axhline(0, linestyle="--", alpha=0.6)

ax.set_xlabel("Window width, Δx")
ax.set_ylabel("Median signed Spearman ρ")
ax.set_title("C. Ordinal association")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8)

fig.suptitle(
    "Effect of comfort representation on scale dependence with offset-bootstrap uncertainty",
    fontsize=15,
    y=1.05,
)

plt.show()

In [ ]:
def dominance_score(summaries, metric_center):
    rows = []

    # assume all summaries share same delta grid
    deltas = sorted(set.intersection(*[set(s["delta"]) for _, s in summaries]))

    for delta in deltas:
        vals = []

        for label, s in summaries:
            val = s.loc[s["delta"] == delta, metric_center].iloc[0]
            vals.append((label, val))

        best_label, best_val = max(vals, key=lambda x: x[1])

        rows.append({
            "delta": delta,
            "best": best_label,
            "best_value": best_val,
        })

    out = pd.DataFrame(rows)

    return (
        out["best"]
        .value_counts(normalize=True)
        .rename("fraction_best")
        .reset_index()
        .rename(columns={"index": "representation"})
    ), out

In [ ]:
summaries = [
    ("Previous 3 groups", summary_prev3),
    ("7 original categories", summary_7),
    ("Comfort / Neutral / Uncomfortable", summary_side3),
]

print(dominance_score(summaries, "frac_kw_significant_center")[0])
print(dominance_score(summaries, "median_epsilon2_center")[0])
print(dominance_score(summaries, "median_spearman_center")[0])

# ara per HDxX

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from matplotlib import gridspec

# ============================================================
# CONFIG
# ============================================================

CSV_PATH = r"../../data/all_surveys(votes).csv"

#  environmental variable

# X_COL = "<T-T_fixed+<T>>"
X_COL = "<HDX-HDX_fixed+<HDX>>"

COMFORT_COL = "thermal_comfort"

ANALYSIS_MODE = "3_groups_temperature"
# ANALYSIS_MODE = "3_groups_HDX"

# If X_COL = "T_K" and values are Kelvin, convert automatically to Celsius
AUTO_CONVERT_KELVIN = True

# Scale analysis parameters
MIN_N = 30
MIN_CLASS_N = 3
N_OFFSETS_PER_DELTA = 25

# For temperature, these values are usually reasonable
# DELTA_MIN = 0.5
# DELTA_STEP = 0.25
# DELTA_MAX_CAP = 12

# For HDX, you may prefer:
DELTA_MIN = 1.0
DELTA_STEP = 0.5
DELTA_MAX_CAP = 20

# Comfort grouping
comfort_to_group = {
    "Very comfortable": "1. Acceptable / neutral",
    "Comfortable": "1. Acceptable / neutral",
    "Slightly comfortable": "1. Acceptable / neutral",
    "Neutral": "1. Acceptable / neutral",

    "Slightly uncomfortable": "2. Discomfort",
    "Uncomfortable": "2. Discomfort",

    "Very uncomfortable": "3. Extreme discomfort",
}

group_order = [
    "1. Acceptable / neutral",
    "2. Discomfort",
    "3. Extreme discomfort",
]

group_score = {
    "1. Acceptable / neutral": 1,
    "2. Discomfort": 2,
    "3. Extreme discomfort": 3,
}

# preparem les dades per a fer l'analisi

In [ ]:
# ============================================================
# LOAD DATA
# ============================================================

df_votes = pd.read_csv(CSV_PATH)

df_scale = df_votes.copy()

df_scale[X_COL] = pd.to_numeric(df_scale[X_COL], errors="coerce")

x_col_used = X_COL

# Convert Kelvin to Celsius if needed
if AUTO_CONVERT_KELVIN and X_COL == "T_K":
    if df_scale[X_COL].median() > 100:
        df_scale["T_C"] = df_scale[X_COL] - 273.15
        x_col_used = "T_C"
    else:
        x_col_used = X_COL

# 3-group comfort
df_scale["comfort_group"] = df_scale[COMFORT_COL].map(comfort_to_group)
df_scale["comfort_score"] = df_scale["comfort_group"].map(group_score)

df_scale = df_scale.dropna(
    subset=[x_col_used, "comfort_group", "comfort_score"]
).copy()

df_scale = df_scale[df_scale["comfort_group"].isin(group_order)].copy()

print("Environmental variable used:", x_col_used)
print("Analysis mode:", ANALYSIS_MODE)
print("N:", len(df_scale))

print("\nComfort group counts:")
print(df_scale["comfort_group"].value_counts().reindex(group_order))

print("\nEnvironmental variable summary:")
print(df_scale[x_col_used].describe().round(3))

In [ ]:
def analyze_one_tiling_scale(
    df,
    x_col,
    label_col,
    score_col,
    class_order,
    delta_x,
    offset=0.0,
    min_n=30,
    min_class_n=3,
):
    """
    Creates non-overlapping windows of width delta_x.
    The offset shifts the bin origin, allowing many tilings.
    """

    x_min = df[x_col].min()
    x_max = df[x_col].max()

    start = x_min - delta_x + offset
    edges = np.arange(start, x_max + 2 * delta_x, delta_x)

    d = df.copy()

    d["x_window"] = pd.cut(
        d[x_col],
        bins=edges,
        include_lowest=True,
        right=False,
    )

    rows = []

    for window, sub in d.groupby("x_window", observed=False):
        if sub.empty:
            continue

        low = window.left
        high = window.right
        center = 0.5 * (low + high)
        n = len(sub)

        counts = (
            sub[label_col]
            .value_counts()
            .reindex(class_order)
            .fillna(0)
        )

        present_classes = counts[counts > 0].index.tolist()
        usable_classes = counts[counts >= min_class_n].index.tolist()

        p_counts = counts[counts > 0] / counts.sum()
        entropy = -(p_counts * np.log(p_counts)).sum()

        valid_test = (n >= min_n) and (len(usable_classes) >= 2)

        H = np.nan
        p_kw = np.nan
        epsilon2 = np.nan
        rho = np.nan
        p_spear = np.nan
        mean_spread = np.nan
        spread_over_iqr = np.nan
        spread_over_sd = np.nan

        if valid_test:
            groups = [
                sub.loc[sub[label_col] == c, x_col].dropna().values
                for c in usable_classes
            ]

            H, p_kw = stats.kruskal(*groups)

            N = sum(len(g) for g in groups)
            k = len(groups)

            epsilon2 = (H - k + 1) / (N - k) if N > k else np.nan

            rho, p_spear = stats.spearmanr(
                sub[x_col],
                sub[score_col],
                nan_policy="omit",
            )

            means = (
                sub.groupby(label_col, observed=False)[x_col]
                .mean()
                .reindex(class_order)
            )

            means = means.loc[
                [c for c in means.index if counts.get(c, 0) >= min_class_n]
            ].dropna()

            if len(means) >= 2:
                mean_spread = means.max() - means.min()

                x_iqr = sub[x_col].quantile(0.75) - sub[x_col].quantile(0.25)
                x_sd = sub[x_col].std()

                spread_over_iqr = mean_spread / x_iqr if x_iqr > 0 else np.nan
                spread_over_sd = mean_spread / x_sd if x_sd > 0 else np.nan

        rows.append({
            "delta_x": delta_x,
            "offset": offset,
            "window": str(window),
            "low": low,
            "high": high,
            "center": center,
            "n": n,
            "n_present_classes": len(present_classes),
            "n_usable_classes": len(usable_classes),
            "entropy": entropy,
            "valid_test": valid_test,
            "kw_H": H,
            "kw_p": p_kw,
            "epsilon2": epsilon2,
            "spearman_rho": rho,
            "spearman_p": p_spear,
            "mean_spread": mean_spread,
            "spread_over_iqr": spread_over_iqr,
            "spread_over_sd": spread_over_sd,
        })

    return pd.DataFrame(rows)

In [ ]:
# ============================================================
# RUN SCALE ANALYSIS
# ============================================================

x_range = df_scale[x_col_used].max() - df_scale[x_col_used].min()

delta_values = np.arange(
    DELTA_MIN,
    min(0.95 * x_range, DELTA_MAX_CAP) + 0.001,
    DELTA_STEP,
)

all_results = []

for delta in delta_values:
    offsets = np.linspace(0, delta, N_OFFSETS_PER_DELTA, endpoint=False)

    for offset in offsets:
        out = analyze_one_tiling_scale(
            df=df_scale,
            x_col=x_col_used,
            label_col="comfort_group",
            score_col="comfort_score",
            class_order=group_order,
            delta_x=delta,
            offset=offset,
            min_n=MIN_N,
            min_class_n=MIN_CLASS_N,
        )

        all_results.append(out)

scale_offsets = pd.concat(all_results, ignore_index=True)

print("scale_offsets shape:", scale_offsets.shape)
scale_offsets.head(10)
scale_offsets_HDX = scale_offsets.copy()

In [ ]:
# ============================================================
# SUMMARY BY SCALE
# ============================================================

valid = scale_offsets[scale_offsets["valid_test"]].copy()

valid["minus_log10_p"] = -np.log10(
    valid["kw_p"].clip(lower=1e-12)
)

scale_summary = (
    valid
    .groupby("delta_x")
    .agg(
        n_windows=("kw_p", "count"),

        frac_kw_significant=("kw_p", lambda x: np.mean(x < 0.05)),

        mean_kw_p=("kw_p", "mean"),
        median_kw_p=("kw_p", "median"),
        q25_kw_p=("kw_p", lambda x: np.nanquantile(x, 0.25)),
        q75_kw_p=("kw_p", lambda x: np.nanquantile(x, 0.75)),

        mean_minus_log10_p=("minus_log10_p", "mean"),
        median_minus_log10_p=("minus_log10_p", "median"),
        q25_minus_log10_p=("minus_log10_p", lambda x: np.nanquantile(x, 0.25)),
        q75_minus_log10_p=("minus_log10_p", lambda x: np.nanquantile(x, 0.75)),

        median_kw_H=("kw_H", "median"),
        mean_kw_H=("kw_H", "mean"),

        median_epsilon2=("epsilon2", "median"),
        mean_epsilon2=("epsilon2", "mean"),

        median_spearman=("spearman_rho", "median"),
        mean_spearman=("spearman_rho", "mean"),
        q25_spearman=("spearman_rho", lambda x: np.nanquantile(x, 0.25)),
        q75_spearman=("spearman_rho", lambda x: np.nanquantile(x, 0.75)),

        median_abs_spearman=("spearman_rho", lambda x: np.nanmedian(np.abs(x))),
        frac_positive_spearman=("spearman_rho", lambda x: np.mean(x > 0)),
    )
    .reset_index()
)

scale_summary.round(4)
scale_summary.to_csv("csvs/scale_summary_HDX.csv", index=False)

# 2x2 plot rellevant

In [ ]:
fig, axes = plt.subplots(
    2, 2,
    figsize=(13, 9),
    constrained_layout=True,
)

# -----------------------------
# A. Fraction significant
# -----------------------------
ax = axes[0, 0]
ax.plot(
    scale_summary["delta_x"],
    scale_summary["frac_kw_significant"],
    marker="o",
)
ax.axhline(0.5, linestyle="--", alpha=0.7, label="50%")
ax.set_xlabel(f"{x_col_used} window width, Δx")
ax.set_ylabel("Fraction Kruskal p < 0.05")
ax.set_title("A. Statistical detectability")
ax.grid(True, alpha=0.3)
ax.legend()

# -----------------------------
# B. Median -log10(p)
# -----------------------------
ax = axes[0, 1]
x = scale_summary["delta_x"]
y = scale_summary["median_minus_log10_p"]
lo = scale_summary["q25_minus_log10_p"]
hi = scale_summary["q75_minus_log10_p"]

ax.plot(x, y, marker="o", label="median")
ax.fill_between(x, lo, hi, alpha=0.2, label="IQR")
ax.axhline(-np.log10(0.05), linestyle="--", alpha=0.7, label="p = 0.05")
ax.set_xlabel(f"{x_col_used} window width, Δx")
ax.set_ylabel("Median -log10(Kruskal p)")
ax.set_title("B. Strength of Kruskal evidence")
ax.grid(True, alpha=0.3)
ax.legend()

# -----------------------------
# C. Epsilon squared
# -----------------------------
ax = axes[1, 0]
ax.plot(
    scale_summary["delta_x"],
    scale_summary["median_epsilon2"],
    marker="o",
)
ax.set_xlabel(f"{x_col_used} window width, Δx")
ax.set_ylabel(r"Median $\epsilon^2$")
ax.set_title("C. Kruskal effect size")
ax.grid(True, alpha=0.3)

# -----------------------------
# D. Spearman with sign
# -----------------------------
ax = axes[1, 1]
x = scale_summary["delta_x"]
y = scale_summary["median_spearman"]
lo = scale_summary["q25_spearman"]
hi = scale_summary["q75_spearman"]

ax.plot(x, y, marker="o", label="median signed ρ")
ax.fill_between(x, lo, hi, alpha=0.2, label="IQR")
ax.axhline(0, linestyle="--", alpha=0.7)
ax.set_xlabel(f"{x_col_used} window width, Δx")
ax.set_ylabel("Spearman ρ")
ax.set_title("D. Ordinal association with sign")
ax.grid(True, alpha=0.3)
ax.legend()

fig.suptitle(
    f"Scale dependence of {x_col_used}-comfort structure ({ANALYSIS_MODE})",
    fontsize=15,
)

plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), constrained_layout=True)

# A. Kruskal fraction
ax = axes[0]
ax.plot(
    scale_summary["delta_x"],
    scale_summary["frac_kw_significant"],
    marker="o",
    label="Previous 3 groups",
)
# ax.plot(
#     scale_summary_side3["delta_x"],
#     scale_summary_side3["frac_kw_significant"],
#     marker="o",
#     label="Comfort / Neutral / Discomfort",
# )

ax.set_xlabel("Window width, Δx")
ax.set_ylabel("Fraction p < 0.05")
ax.set_title("A. Statistical detectability")
ax.grid(True, alpha=0.3)
ax.legend()

# B. epsilon2
ax = axes[1]
ax.plot(
    scale_summary["delta_x"],
    scale_summary["median_epsilon2"],
    marker="o",
    label="Previous 3 groups",
)
# ax.plot(
#     scale_summary_side3["delta_x"],
#     scale_summary_side3["median_epsilon2"],
#     marker="o",
#     label="Comfort / Neutral / Discomfort",
# )
ax.set_xlabel("Window width, Δx")
ax.set_ylabel(r"Median $\epsilon^2$")
ax.set_title("B. Effect size")
ax.grid(True, alpha=0.3)
ax.legend()

# C. signed Spearman
ax = axes[2]
ax.plot(
    scale_summary["delta_x"],
    scale_summary["median_spearman"],
    marker="o",
    label="Previous 3 groups",
)
# ax.plot(
#     scale_summary_side3["delta_x"],
#     scale_summary_side3["median_spearman"],
#     marker="o",
#     label="Comfort / Neutral / Discomfort",
# )
ax.axhline(0, linestyle="--", alpha=0.6)
ax.set_xlabel("Window width, Δx")
ax.set_ylabel("Median signed Spearman ρ")
ax.set_title("C. Ordinal association")
ax.grid(True, alpha=0.3)
ax.legend()

fig.suptitle(
    f"Effect of comfort grouping on scale dependence ({x_col_used})",
    fontsize=15,
)

plt.show()

scale_summary_HDX = scale_summary.copy()

In [ ]:
# ============================================================
# SCALE-SPACE HEATMAP
# ============================================================

center_bin_width = DELTA_STEP * 2
min_cell_support = 1
clip_p_min = 1e-12

# Optional reference lines
# For temperature, put around 29 if relevant
transition_lines = []

if "T" in x_col_used:
    transition_lines = [29]
elif "HDX" in x_col_used:
    transition_lines = [32, 37.5]

heat = scale_offsets[scale_offsets["valid_test"]].copy()

heat["minus_log10_p"] = -np.log10(
    heat["kw_p"].clip(lower=clip_p_min)
)

heat["center_bin"] = (
    np.round(heat["center"] / center_bin_width) * center_bin_width
)

pivot_value = heat.pivot_table(
    index="delta_x",
    columns="center_bin",
    values="minus_log10_p",
    aggfunc="median",
)

pivot_support = heat.pivot_table(
    index="delta_x",
    columns="center_bin",
    values="minus_log10_p",
    aggfunc="count",
)

pivot_value = pivot_value.sort_index().sort_index(axis=1)
pivot_support = pivot_support.reindex_like(pivot_value)

pivot_masked = pivot_value.mask(pivot_support < min_cell_support)

# Edges for pcolormesh
x_centers = pivot_masked.columns.to_numpy(dtype=float)
y_centers = pivot_masked.index.to_numpy(dtype=float)

x_edges = np.r_[
    x_centers[0] - center_bin_width / 2,
    x_centers + center_bin_width / 2,
]

if len(y_centers) > 1:
    y_mid = 0.5 * (y_centers[:-1] + y_centers[1:])
    y_edges = np.r_[
        y_centers[0] - (y_mid[0] - y_centers[0]),
        y_mid,
        y_centers[-1] + (y_centers[-1] - y_mid[-1]),
    ]
else:
    dy = 0.5
    y_edges = np.array([y_centers[0] - dy, y_centers[0] + dy])

Z = np.ma.masked_invalid(pivot_masked.to_numpy())

# Observed coverage
x_obs = df_scale[x_col_used].dropna().to_numpy()

bin_edges = np.arange(
    x_edges.min(),
    x_edges.max() + center_bin_width,
    center_bin_width,
)

counts, edges = np.histogram(x_obs, bins=bin_edges)
count_centers = 0.5 * (edges[:-1] + edges[1:])

xlim_low = np.nanmin(x_obs) - 0.5
xlim_high = np.nanmax(x_obs) + 0.5

# Plot
fig = plt.figure(figsize=(13, 8))

gs = gridspec.GridSpec(
    2, 2,
    width_ratios=[20, 1],
    height_ratios=[1.2, 4],
    wspace=0.12,
    hspace=0.18,
)

ax_top = fig.add_subplot(gs[0, 0])
ax = fig.add_subplot(gs[1, 0], sharex=ax_top)
cax = fig.add_subplot(gs[1, 1])

ax_empty = fig.add_subplot(gs[0, 1])
ax_empty.axis("off")

# Top histogram
ax_top.bar(
    count_centers,
    counts,
    width=center_bin_width * 0.9,
    align="center",
    alpha=0.75,
)

ax_top.set_ylabel("Votes\nper bin")
ax_top.set_title(f"Observed {x_col_used} coverage", fontsize=12)
ax_top.grid(True, axis="y", alpha=0.3)
ax_top.tick_params(axis="x", labelbottom=True)

for xc in transition_lines:
    ax_top.axvline(
        xc,
        linestyle="--",
        color="black",
        alpha=0.6,
        linewidth=1.2,
    )

# Heatmap
cmap = plt.cm.viridis.copy()
cmap.set_bad(color="white")

mesh = ax.pcolormesh(
    x_edges,
    y_edges,
    Z,
    cmap=cmap,
    shading="auto",
)

cbar = fig.colorbar(mesh, cax=cax)
cbar.set_label("Median -log10(Kruskal p)")

for xc in transition_lines:
    ax.axvline(
        xc,
        linestyle="--",
        color="white",
        alpha=0.85,
        linewidth=1.5,
    )

    ax.text(
        xc,
        y_edges[0] + 0.2,
        str(xc),
        color="white",
        rotation=90,
        va="bottom",
        ha="right",
        fontsize=9,
    )

ax.set_xlabel(f"{x_col_used} window center")
ax.set_ylabel(f"{x_col_used} window width, Δx")
ax.set_title(
    f"Scale-space map of comfort-class statistical detectability ({ANALYSIS_MODE})",
    fontsize=12,
)

ax.set_xlim(xlim_low, xlim_high)
ax.set_ylim(y_edges[0], y_edges[-1])

fig.suptitle(
    f"Observed {x_col_used} coverage and scale-space statistical detectability",
    fontsize=15,
    y=0.98,
)

plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# CONFIG
# ============================================================

T_COL = "<T-T_fixed+<T>>"
HDX_COL = "<HDX-HDX_fixed+<HDX>>"

# These should already exist from your analyses
# scale_summary_T
# scale_summary_HDX
# df_votes or df_scale

def get_delta_col(summary):
    if "delta_x" in summary.columns:
        return "delta_x"
    elif "delta_hdx" in summary.columns:
        return "delta_hdx"
    elif "delta" in summary.columns:
        return "delta"
    else:
        raise ValueError("No delta column found.")


def add_normalized_scale(summary, df, x_col, label):
    s = summary.copy()
    delta_col = get_delta_col(s)

    x = pd.to_numeric(df[x_col], errors="coerce").dropna()
    iqr = x.quantile(0.75) - x.quantile(0.25)

    s["delta_norm"] = s[delta_col] / iqr
    s["variable"] = label
    s["IQR"] = iqr

    return s


summary_T_norm = add_normalized_scale(
    scale_summary_T,
    df_votes,
    T_COL,
    "Temperature",
)

summary_HDX_norm = add_normalized_scale(
    scale_summary_HDX,
    df_votes,
    HDX_COL,
    "HDX",
)

print("Temperature IQR:", summary_T_norm["IQR"].iloc[0])
print("HDX IQR:", summary_HDX_norm["IQR"].iloc[0])

In [ ]:
fig, axes = plt.subplots(
    2, 2,
    figsize=(13, 9),
    constrained_layout=True,
)

summaries = [
    ("Temperature", summary_T_norm),
    ("HDX", summary_HDX_norm),
]

# ============================================================
# A. Statistical detectability
# ============================================================

ax = axes[0, 0]

for label, s in summaries:
    ax.plot(
        s["delta_norm"],
        s["frac_kw_significant"],
        marker="o",
        label=label,
    )

ax.axhline(0.3, linestyle="--", alpha=0.6, label="30%")
ax.axhline(0.5, linestyle="--", alpha=0.6, label="50%")

ax.set_xlabel(r"Window width / variable IQR, $\Delta x / IQR(x)$")
ax.set_ylabel("Fraction Kruskal p < 0.05")
ax.set_title("A. Statistical detectability")
ax.grid(True, alpha=0.3)
ax.legend()


# ============================================================
# B. Strength of Kruskal evidence
# ============================================================

ax = axes[0, 1]

for label, s in summaries:
    ax.plot(
        s["delta_norm"],
        s["median_minus_log10_p"],
        marker="o",
        label=label,
    )

    if {"q25_minus_log10_p", "q75_minus_log10_p"}.issubset(s.columns):
        ax.fill_between(
            s["delta_norm"],
            s["q25_minus_log10_p"],
            s["q75_minus_log10_p"],
            alpha=0.18,
        )

ax.axhline(
    -np.log10(0.05),
    linestyle="--",
    alpha=0.7,
    label="p = 0.05",
)

ax.set_xlabel(r"Window width / variable IQR, $\Delta x / IQR(x)$")
ax.set_ylabel(r"Median $-\log_{10}$(Kruskal p)")
ax.set_title("B. Strength of Kruskal evidence")
ax.grid(True, alpha=0.3)
ax.legend()


# ============================================================
# C. Kruskal effect size
# ============================================================

ax = axes[1, 0]

for label, s in summaries:
    ax.plot(
        s["delta_norm"],
        s["median_epsilon2"],
        marker="o",
        label=label,
    )

ax.set_xlabel(r"Window width / variable IQR, $\Delta x / IQR(x)$")
ax.set_ylabel(r"Median $\epsilon^2$")
ax.set_title("C. Kruskal effect size")
ax.grid(True, alpha=0.3)
ax.legend()


# ============================================================
# D. Ordinal association
# ============================================================

ax = axes[1, 1]

for label, s in summaries:
    ax.plot(
        s["delta_norm"],
        s["median_spearman"],
        marker="o",
        label=label,
    )

    if {"q25_spearman", "q75_spearman"}.issubset(s.columns):
        ax.fill_between(
            s["delta_norm"],
            s["q25_spearman"],
            s["q75_spearman"],
            alpha=0.18,
        )

ax.axhline(0, linestyle="--", alpha=0.7)

ax.set_xlabel(r"Window width / variable IQR, $\Delta x / IQR(x)$")
ax.set_ylabel(r"Median signed Spearman $\rho$")
ax.set_title("D. Ordinal association")
ax.grid(True, alpha=0.3)
ax.legend()

fig.suptitle(
    "Temperature vs HDX scale dependence in normalized units",
    fontsize=15,
)

plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# CONFIG
# ============================================================

T_COL = "<T-T_fixed+<T>>"
HDX_COL = "<HDX-HDX_fixed+<HDX>>"

N_BOOT = 1000
CI = (2.5, 97.5)
RANDOM_STATE = 123


# ============================================================
# CHECK / HELPERS
# ============================================================

def get_delta_col(df):
    if "delta_x" in df.columns:
        return "delta_x"
    elif "delta_hdx" in df.columns:
        return "delta_hdx"
    elif "delta" in df.columns:
        return "delta"
    else:
        raise ValueError("No delta_x, delta_hdx or delta column found.")


def check_scale_offset(df, name):
    print(f"\n{name}")
    print("Shape:", df.shape)
    print("Delta column:", get_delta_col(df))
    
    needed = ["offset", "valid_test", "kw_p", "epsilon2", "spearman_rho"]
    for c in needed:
        print(c, "OK" if c in df.columns else "MISSING")


check_scale_offset(scale_offsets_T, "scale_offset_T")
check_scale_offset(scale_offsets_HDX, "scale_offset_HDX")

In [ ]:
# ============================================================
# BOOTSTRAP FUNCTIONS
# ============================================================

def compute_metrics(d):
    kw_p = d["kw_p"].dropna().values
    eps = d["epsilon2"].dropna().values
    rho = d["spearman_rho"].dropna().values

    return {
        "frac_kw_significant": np.mean(kw_p < 0.05) if len(kw_p) else np.nan,
        "median_minus_log10_p": (
            np.nanmedian(-np.log10(np.clip(kw_p, 1e-12, None)))
            if len(kw_p) else np.nan
        ),
        "median_epsilon2": np.nanmedian(eps) if len(eps) else np.nan,
        "median_spearman": np.nanmedian(rho) if len(rho) else np.nan,
    }


def bootstrap_offset_summary(
    scale_df,
    label,
    n_boot=1000,
    ci=(2.5, 97.5),
    random_state=123,
):
    rng = np.random.default_rng(random_state)

    d = scale_df[scale_df["valid_test"]].copy()
    delta_col = get_delta_col(d)

    rows = []

    for delta, d_delta in d.groupby(delta_col):
        # Central line: all valid windows together
        central = compute_metrics(d_delta)

        # Bootstrap over offsets
        offset_groups = [
            g.copy()
            for _, g in d_delta.groupby("offset")
            if len(g) > 0
        ]

        n_offsets = len(offset_groups)

        boot_vals = {
            "frac_kw_significant": [],
            "median_minus_log10_p": [],
            "median_epsilon2": [],
            "median_spearman": [],
        }

        if n_offsets >= 2:
            for _ in range(n_boot):
                sampled_ids = rng.integers(0, n_offsets, size=n_offsets)

                sampled = pd.concat(
                    [offset_groups[i] for i in sampled_ids],
                    ignore_index=True,
                )

                m = compute_metrics(sampled)

                for key in boot_vals:
                    boot_vals[key].append(m[key])

        row = {
            "delta": delta,
            "variable": label,
            "n_windows": len(d_delta),
            "n_offsets": n_offsets,
        }

        for key, value in central.items():
            row[f"{key}_center"] = value

            if n_offsets >= 2:
                row[f"{key}_low"] = np.nanpercentile(boot_vals[key], ci[0])
                row[f"{key}_high"] = np.nanpercentile(boot_vals[key], ci[1])
            else:
                row[f"{key}_low"] = np.nan
                row[f"{key}_high"] = np.nan

        rows.append(row)

    return pd.DataFrame(rows).sort_values("delta")


def add_delta_norm(summary_boot, df, x_col):
    s = summary_boot.copy()

    x = pd.to_numeric(df[x_col], errors="coerce").dropna()
    iqr = x.quantile(0.75) - x.quantile(0.25)

    s["IQR"] = iqr
    s["delta_norm"] = s["delta"] / iqr

    return s

In [ ]:
# ============================================================
# RUN BOOTSTRAP
# ============================================================

summary_T_boot = bootstrap_offset_summary(
    scale_df=scale_offsets_T,
    label="Temperature",
    n_boot=N_BOOT,
    ci=CI,
    random_state=RANDOM_STATE,
)

summary_HDX_boot = bootstrap_offset_summary(
    scale_df=scale_offsets_HDX,
    label="HDX",
    n_boot=N_BOOT,
    ci=CI,
    random_state=RANDOM_STATE,
)

summary_T_boot = add_delta_norm(
    summary_T_boot,
    df_votes,
    T_COL,
)

summary_HDX_boot = add_delta_norm(
    summary_HDX_boot,
    df_votes,
    HDX_COL,
)

print("Temperature IQR:", summary_T_boot["IQR"].iloc[0])
print("HDX IQR:", summary_HDX_boot["IQR"].iloc[0])

display(summary_T_boot.head())
display(summary_HDX_boot.head())

In [ ]:
# ============================================================
# PLOT: T vs HDX, normalized scale, bootstrap uncertainty
# ============================================================

fig, axes = plt.subplots(
    2, 2,
    figsize=(13, 9),
    constrained_layout=True,
)

summaries = [
    ("Temperature", summary_T_boot),
    ("HDX", summary_HDX_boot),
]

# ------------------------------------------------------------
# A. Statistical detectability
# ------------------------------------------------------------

ax = axes[0, 0]

for label, s in summaries:
    ax.plot(
        s["delta_norm"],
        s["frac_kw_significant_center"],
        marker="o",
        label=label,
    )
    ax.fill_between(
        s["delta_norm"],
        s["frac_kw_significant_low"],
        s["frac_kw_significant_high"],
        alpha=0.16,
    )




ax.set_xlabel(r"Window width / variable IQR, $\Delta x / IQR(x)$")
ax.set_ylabel("Fraction Kruskal p < 0.05")
ax.set_title("A. Statistical detectability")
ax.grid(True, alpha=0.3)
ax.legend()


# ------------------------------------------------------------
# B. Strength of Kruskal evidence
# ------------------------------------------------------------

ax = axes[0, 1]

for label, s in summaries:
    ax.plot(
        s["delta_norm"],
        s["median_minus_log10_p_center"],
        marker="o",
        label=label,
    )
    ax.fill_between(
        s["delta_norm"],
        s["median_minus_log10_p_low"],
        s["median_minus_log10_p_high"],
        alpha=0.16,
    )

ax.axhline(
    -np.log10(0.05),
    linestyle="--",
    alpha=0.7,
    label="p = 0.05",
)

ax.set_xlabel(r"Window width / variable IQR, $\Delta x / IQR(x)$")
ax.set_ylabel(r"Median $-\log_{10}$(Kruskal p)")
ax.set_title("B. Strength of Kruskal evidence")
ax.grid(True, alpha=0.3)
ax.legend()


# ------------------------------------------------------------
# C. Kruskal effect size
# ------------------------------------------------------------

ax = axes[1, 0]

for label, s in summaries:
    ax.plot(
        s["delta_norm"],
        s["median_epsilon2_center"],
        marker="o",
        label=label,
    )
    ax.fill_between(
        s["delta_norm"],
        s["median_epsilon2_low"],
        s["median_epsilon2_high"],
        alpha=0.16,
    )

ax.set_xlabel(r"Window width / variable IQR, $\Delta x / IQR(x)$")
ax.set_ylabel(r"Median $\epsilon^2$")
ax.set_title("C. Kruskal effect size")
ax.grid(True, alpha=0.3)
ax.legend()


# ------------------------------------------------------------
# D. Ordinal association
# ------------------------------------------------------------

ax = axes[1, 1]

for label, s in summaries:
    ax.plot(
        s["delta_norm"],
        s["median_spearman_center"],
        marker="o",
        label=label,
    )
    ax.fill_between(
        s["delta_norm"],
        s["median_spearman_low"],
        s["median_spearman_high"],
        alpha=0.16,
    )

ax.axhline(0, linestyle="--", alpha=0.7)

ax.set_xlabel(r"Window width / variable IQR, $\Delta x / IQR(x)$")
ax.set_ylabel(r"Median signed Spearman $\rho$")
# ax.set_ylim(0, 3.5)
ax.set_title("D. Ordinal association")
ax.grid(True, alpha=0.3)
ax.legend()

# fig.suptitle(
#     "Temperature vs HDX scale dependence in normalized units\nwith offset-bootstrap uncertainty",
#     fontsize=15,
# )

plt.show()

# Llavors separa millor el HDX que la T (i per tant pitjor separació ordinal a finestres petites?)

Mirem aquesta separació bé

In [ ]:
from sklearn.metrics import normalized_mutual_info_score

T_COL = "<T-T_fixed+<T>>"
HDX_COL = "<HDX-HDX_fixed+<HDX>>"

def walk_variable_nmi(df, x_col, walk_col=None, q=10):
    d = df.copy()

    if walk_col is None:
        if "walk_id" in d.columns:
            walk_col = "walk_id"
        elif "walk" in d.columns:
            walk_col = "walk"
        elif "space_code" in d.columns:
            d["walk_tmp"] = d["space_code"].astype(str).str[:2]
            walk_col = "walk_tmp"
        else:
            raise ValueError("No walk column found.")

    d[x_col] = pd.to_numeric(d[x_col], errors="coerce")
    d = d[[walk_col, x_col]].dropna().copy()

    d["x_bin"] = pd.qcut(
        d[x_col],
        q=q,
        duplicates="drop",
        labels=False,
    )

    nmi = normalized_mutual_info_score(
        d[walk_col].astype(str),
        d["x_bin"].astype(str),
    )

    return {
        "variable": x_col,
        "q_bins": q,
        "normalized_mutual_information": nmi,
    }


nmi_results = pd.DataFrame([
    walk_variable_nmi(df_votes, "<T-T_fixed+<T>>", q=10),
    walk_variable_nmi(df_votes, "<HDX-HDX_fixed+<HDX>>", q=10),
])

nmi_results.round(4)

In [ ]:
def walk_region_purity(df, x_col, walk_col=None, q=3):
    d = df.copy()

    if walk_col is None:
        if "walk_id" in d.columns:
            walk_col = "walk_id"
        elif "walk" in d.columns:
            walk_col = "walk"
        elif "space_code" in d.columns:
            d["walk_tmp"] = d["space_code"].astype(str).str[:2]
            walk_col = "walk_tmp"
        else:
            raise ValueError("No walk column found.")

    d[x_col] = pd.to_numeric(d[x_col], errors="coerce")
    d = d[[walk_col, x_col]].dropna().copy()

    d["x_regime"] = pd.qcut(
        d[x_col],
        q=q,
        duplicates="drop",
    )

    tab = pd.crosstab(
        d[walk_col],
        d["x_regime"],
        normalize="index",
    )

    purity = tab.max(axis=1)

    out = pd.DataFrame({
        "walk": purity.index,
        "purity": purity.values,
    })

    return {
        "variable": x_col,
        "q": q,
        "mean_walk_purity": purity.mean(),
        "median_walk_purity": purity.median(),
        "q25_walk_purity": purity.quantile(0.25),
        "q75_walk_purity": purity.quantile(0.75),
    }, out, tab


purity_T, purity_by_walk_T, tab_T = walk_region_purity(
    df_votes,
    T_COL,
    q=3,
)

purity_HDX, purity_by_walk_HDX, tab_HDX = walk_region_purity(
    df_votes,
    HDX_COL,
    q=3,
)

purity_results = pd.DataFrame([purity_T, purity_HDX])
purity_results.round(4)

In [ ]:
import matplotlib.pyplot as plt

def walk_bin_heatmap_data(df, x_col, walk_col=None, q=12):
    d = df.copy()

    if walk_col is None:
        if "walk_id" in d.columns:
            walk_col = "walk_id"
        elif "walk" in d.columns:
            walk_col = "walk"
        elif "space_code" in d.columns:
            d["walk_tmp"] = d["space_code"].astype(str).str[:2]
            walk_col = "walk_tmp"
        else:
            raise ValueError("No walk column found.")

    d[x_col] = pd.to_numeric(d[x_col], errors="coerce")
    d = d[[walk_col, x_col]].dropna().copy()

    d["x_bin"] = pd.qcut(
        d[x_col],
        q=q,
        duplicates="drop",
    )

    # Normalize by row: P(bin | walk)
    tab = pd.crosstab(
        d[walk_col],
        d["x_bin"],
        normalize="index",
    )

    # Sort walks by mean x value
    walk_means = d.groupby(walk_col)[x_col].mean()
    tab = tab.loc[walk_means.sort_values().index]

    bin_labels = [
        f"{interval.left:.1f}-{interval.right:.1f}"
        for interval in tab.columns
    ]

    return tab, bin_labels


tab_T_heat, labels_T = walk_bin_heatmap_data(
    df_votes,
    T_COL,
    q=12,
)

tab_HDX_heat, labels_HDX = walk_bin_heatmap_data(
    df_votes,
    HDX_COL,
    q=12,
)

fig, axes = plt.subplots(
    1, 2,
    figsize=(15, 8),
    constrained_layout=True,
)

# Temperature heatmap
im0 = axes[0].imshow(
    tab_T_heat.values,
    aspect="auto",
    interpolation="nearest",
    vmin=0,
    vmax=1,
)

axes[0].set_title("Walk distribution across temperature bins")
axes[0].set_xlabel("Temperature quantile bin")
axes[0].set_ylabel("Walks sorted by mean temperature")
axes[0].set_xticks(range(len(labels_T)))
axes[0].set_xticklabels(labels_T, rotation=45, ha="right", fontsize=8)

# HDX heatmap
im1 = axes[1].imshow(
    tab_HDX_heat.values,
    aspect="auto",
    interpolation="nearest",
    vmin=0,
    vmax=1,
)

axes[1].set_title("Walk distribution across HDX bins")
axes[1].set_xlabel("HDX quantile bin")
axes[1].set_ylabel("Walks sorted by mean HDX")
axes[1].set_xticks(range(len(labels_HDX)))
axes[1].set_xticklabels(labels_HDX, rotation=45, ha="right", fontsize=8)

fig.colorbar(im1, ax=axes, label="P(bin | walk)")

plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def walk_median_iqr_summary(df, x_col, walk_col=None):
    d = df.copy()

    if walk_col is None:
        if "walk_id" in d.columns:
            walk_col = "walk_id"
        elif "walk" in d.columns:
            walk_col = "walk"
        elif "space_code" in d.columns:
            d["walk_tmp"] = d["space_code"].astype(str).str[:2]
            walk_col = "walk_tmp"
        else:
            raise ValueError("No walk column found.")

    d[x_col] = pd.to_numeric(d[x_col], errors="coerce")
    d = d[[walk_col, x_col]].dropna().copy()

    out = (
        d
        .groupby(walk_col)[x_col]
        .agg(
            n="count",
            mean="mean",
            median="median",
            q25=lambda x: x.quantile(0.25),
            q75=lambda x: x.quantile(0.75),
            min="min",
            max="max",
        )
        .reset_index()
        .sort_values("median")
    )

    out["iqr"] = out["q75"] - out["q25"]

    return out


T_COL = "<T-T_fixed+<T>>"
HDX_COL = "<HDX-HDX_fixed+<HDX>>"

walk_T = walk_median_iqr_summary(df_votes, T_COL)
walk_HDX = walk_median_iqr_summary(df_votes, HDX_COL)

fig, axes = plt.subplots(
    1, 2,
    figsize=(14, 5),
    constrained_layout=True,
)

for ax, w, label in [
    (axes[0], walk_T, "Temperature"),
    (axes[1], walk_HDX, "HDX"),
]:
    x = np.arange(len(w))

    y = w["median"].to_numpy()

    yerr_low = y - w["q25"].to_numpy()
    yerr_high = w["q75"].to_numpy() - y

    # Numerical safety
    yerr_low = np.clip(yerr_low, 0, None)
    yerr_high = np.clip(yerr_high, 0, None)

    yerr = np.vstack([yerr_low, yerr_high])

    ax.errorbar(
        x,
        y,
        yerr=yerr,
        marker="o",
        linestyle="none",
        capsize=3,
    )
    if label == "HDX":
        ax.axhline(32, linestyle="--", alpha=0.7,label="32 HDX")
        ax.axhline(37.5, linestyle="--", alpha=0.7,label="37.5 HDX")
    else:
        ax.axhline(27, linestyle="--", alpha=0.7,label="27 °C")
        ax.axhline(30.5, linestyle="--", alpha=0.7,label="30.5 °C")

    ax.set_title(f"Walk-level {label}: median ± IQR")
    ax.set_xlabel("Walks sorted by median value")
    ax.set_ylabel(label)
    ax.grid(True, alpha=0.3)
    
    ax.legend()

plt.show()

In [ ]:
from sklearn.cluster import KMeans
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# walk_HDX should come from your walk_median_iqr_summary
X = walk_HDX[["median"]].to_numpy()

kmeans = KMeans(n_clusters=3, random_state=0, n_init=20)
walk_HDX["HDX_walk_cluster"] = kmeans.fit_predict(X)

# Order clusters by median HDX
cluster_order = (
    walk_HDX
    .groupby("HDX_walk_cluster")["median"]
    .mean()
    .sort_values()
    .index
)

cluster_map = {
    old: new for new, old in enumerate(cluster_order)
}

walk_HDX["HDX_walk_region"] = walk_HDX["HDX_walk_cluster"].map(cluster_map)

print(
    walk_HDX
    .groupby("HDX_walk_region")["median"]
    .agg(["count", "min", "median", "max"])
    .round(3)
)

plt.figure(figsize=(8, 5))

for region, sub in walk_HDX.groupby("HDX_walk_region"):
    plt.scatter(
        sub.index,
        sub["median"],
        label=f"Region {region}",
    )

plt.xlabel("Walk index")
plt.ylabel("Walk median HDX")
plt.title("HDX walk-level regions from 1D clustering")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# Probem a mirar les noves distribucions ammb la separació per règims d'HDX

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

hdx_col = "<HDX-HDX_fixed+<HDX>>"
comfort_col = "thermal_comfort"

# ============================================================
# DEFINE HDX REGIME THRESHOLDS FROM WALK-LEVEL CLUSTERS
# ============================================================

# Centers of the 3 HDX walk regions
region_centers = (
    walk_HDX
    .groupby("HDX_walk_region")["median"]
    .median()
    .sort_values()
)

print("HDX regime centers:")
print(region_centers)

# Boundaries = midpoints between neighboring region centers
c = region_centers.values

b1 = 0.5 * (c[0] + c[1])
b2 = 0.5 * (c[1] + c[2])

print("\nHDX boundaries:")
print("Low / Middle:", round(b1, 3))
print("Middle / High:", round(b2, 3))

# ============================================================
# ASSIGN EACH VOTE TO HDX REGIME
# ============================================================

df_hdx_reg = df_votes.copy()
df_hdx_reg[hdx_col] = pd.to_numeric(df_hdx_reg[hdx_col], errors="coerce")

df_hdx_reg = df_hdx_reg.dropna(subset=[hdx_col, comfort_col]).copy()

df_hdx_reg["HDX_regime"] = pd.cut(
    df_hdx_reg[hdx_col],
    bins=[-np.inf, b1, b2, np.inf],
    labels=["Low HDX", "Middle HDX", "High HDX"],
)

print("\nVotes per HDX regime:")
print(df_hdx_reg["HDX_regime"].value_counts().reindex(["Low HDX", "Middle HDX", "High HDX"]))

## Ara ho fem per les 7 categories originals

In [ ]:
# ============================================================
# 7 ORIGINAL COMFORT CATEGORIES
# ============================================================

comfort_order_7 = [
    "Very comfortable",
    "Comfortable",
    "Slightly comfortable",
    "Neutral",
    "Slightly uncomfortable",
    "Uncomfortable",
    "Very uncomfortable",
]

regime_order = ["Low HDX", "Middle HDX", "High HDX"]

tab_counts_7 = pd.crosstab(
    df_hdx_reg["HDX_regime"],
    df_hdx_reg[comfort_col],
).reindex(
    index=regime_order,
    columns=comfort_order_7,
    fill_value=0,
)

tab_prob_7 = tab_counts_7.div(tab_counts_7.sum(axis=1), axis=0)

print("Counts:")
display(tab_counts_7)

print("Probabilities:")
display(tab_prob_7.round(3))

# ============================================================
# STACKED BAR PLOT
# ============================================================

ax = tab_prob_7.plot(
    kind="bar",
    stacked=True,
    figsize=(10, 5),
)

ax.set_ylabel("Proportion within HDX regime")
ax.set_xlabel("HDX regime")
ax.set_title("Comfort-category distribution within HDX regimes")
ax.grid(True, axis="y", alpha=0.3)
ax.legend(
    title="Thermal comfort",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
)

plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

im = ax.imshow(
    tab_prob_7.T.values,
    aspect="auto",
    interpolation="nearest",
)

ax.set_xticks(range(len(regime_order)))
ax.set_xticklabels(regime_order)

ax.set_yticks(range(len(comfort_order_7)))
ax.set_yticklabels(comfort_order_7)

ax.set_xlabel("HDX regime")
ax.set_ylabel("Thermal comfort category")
ax.set_title("P(comfort category | HDX regime)")

# annotate values
for i in range(tab_prob_7.T.shape[0]):
    for j in range(tab_prob_7.T.shape[1]):
        val = tab_prob_7.T.iloc[i, j]
        ax.text(
            j, i,
            f"{val:.2f}",
            ha="center",
            va="center",
            color="white" if val > 0.25 else "black",
            fontsize=9,
        )

im.set_cmap("Reds")
fig.colorbar(im, ax=ax, label="Probability within HDX regime")
plt.tight_layout()
plt.show()

## Per els 3 altres estats

In [ ]:
# ============================================================
# PREVIOUS 3-GROUP REPRESENTATION
# ============================================================

comfort_to_prev3 = {
    "Very comfortable": "1. Acceptable / neutral",
    "Comfortable": "1. Acceptable / neutral",
    "Slightly comfortable": "1. Acceptable / neutral",
    "Neutral": "1. Acceptable / neutral",

    "Slightly uncomfortable": "2. Discomfort",
    "Uncomfortable": "2. Discomfort",

    "Very uncomfortable": "3. Extreme discomfort",
}

prev3_order = [
    "1. Acceptable / neutral",
    "2. Discomfort",
    "3. Extreme discomfort",
]

df_hdx_reg["comfort_prev3"] = df_hdx_reg[comfort_col].map(comfort_to_prev3)

tab_counts_3 = pd.crosstab(
    df_hdx_reg["HDX_regime"],
    df_hdx_reg["comfort_prev3"],
).reindex(
    index=regime_order,
    columns=prev3_order,
    fill_value=0,
)

tab_prob_3 = tab_counts_3.div(tab_counts_3.sum(axis=1), axis=0)

print("Counts:")
display(tab_counts_3)

print("Probabilities:")
display(tab_prob_3.round(3))

# ============================================================
# STACKED BAR PLOT
# ============================================================

ax = tab_prob_3.plot(
    kind="bar",
    stacked=True,
    figsize=(8, 5),
)

ax.set_ylabel("Proportion within HDX regime")
ax.set_xlabel("HDX regime")
ax.set_title("3-state comfort distribution within HDX regimes")
ax.grid(True, axis="y", alpha=0.3)
ax.legend(
    title="Comfort macrostate",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
)

plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

im = ax.imshow(
    tab_prob_3.T.values,
    aspect="auto",
    interpolation="nearest",
)

ax.set_xticks(range(len(regime_order)))
ax.set_xticklabels(regime_order)

ax.set_yticks(range(len(prev3_order)))
ax.set_yticklabels(prev3_order)

ax.set_xlabel("HDX regime")
ax.set_ylabel("Thermal comfort category")
ax.set_title("P(comfort category | HDX regime)")

# annotate values
for i in range(tab_prob_3.T.shape[0]):
    for j in range(tab_prob_3.T.shape[1]):
        val = tab_prob_3.T.iloc[i, j]
        ax.text(
            j, i,
            f"{val:.2f}",
            ha="center",
            va="center",
            color="white" if val > 0.25 else "black",
            fontsize=9,
        )

im.set_cmap("Reds")
fig.colorbar(im, ax=ax, label="Probability within HDX regime")
plt.tight_layout()
plt.show()